# 第１弾 STEP 2: Iceberg Table の読み取り・スナップショット・タイムトラベル

## 目的
- Iceberg Table からデータを読み取る
- スナップショットを使ったタイムトラベルを確認する
- 特定スナップショット時点のデータを参照する
- Iceberg の変更履歴（change tracking）の仕組みを理解する

## 前提
`01_write_and_structure.ipynb` を先に実行して `ORDERS_ICEBERG` テーブルにデータが入っていること

## STEP 0: 接続確認

In [ ]:
%%sql -r setup_result
USE DATABASE ICEBERG_DB;
USE SCHEMA WORK;
USE WAREHOUSE SANDBOX_WH;

## STEP 1: 通常の読み取り

In [ ]:
%%sql -r order_summary
-- 通常クエリ（最新スナップショットを参照）
SELECT
    YEAR(O_ORDERDATE) AS ORDER_YEAR,
    COUNT(*) AS CNT,
    SUM(O_TOTALPRICE) AS TOTAL_PRICE
FROM ICEBERG_DB.WORK.ORDERS_ICEBERG
GROUP BY 1
ORDER BY 1;

## STEP 2: スナップショット一覧の確認

In [ ]:
import json
from datetime import datetime, timezone
from snowflake.snowpark.context import get_active_session
session = get_active_session()

row = session.sql("SHOW ICEBERG TABLES LIKE 'ORDERS_ICEBERG' IN SCHEMA ICEBERG_DB.WORK").collect()
base_location = row[0]['base_location']
stage_path = f'@ICEBERG_DB.WORK.ICEBERG_WORK_STAGE/{base_location}'

metadata_rows = session.sql(f"""
    SELECT $1 AS metadata_json
    FROM {stage_path}metadata/ (FILE_FORMAT => 'ICEBERG_DB.WORK.JSON_FF', PATTERN => '.*metadata\\\\.json')
    ORDER BY METADATA$FILENAME DESC
    LIMIT 1
""").collect()

metadata = json.loads(str(metadata_rows[0]['METADATA_JSON']))
snapshots = metadata.get('snapshots', [])

def ms_to_ts(ms):
    return datetime.fromtimestamp(ms / 1000, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S.%f')[:-3] + ' +0000'

print(f'スナップショット数: {len(snapshots)}\n')
for i, snap in enumerate(snapshots):
    summary = snap.get('summary', {})
    print(f'[{i+1}] snapshot-id:       {snap["snapshot-id"]}')
    print(f'    sequence-number:    {snap["sequence-number"]}')
    print(f'    timestamp:          {ms_to_ts(snap["timestamp-ms"])}')
    print(f'    operation:          {summary.get("operation", "N/A")}')
    print(f'    total-records:      {summary.get("total-records", "N/A")}')
    parent = snap.get('parent-snapshot-id', '-')
    print(f'    parent-snapshot-id: {parent}')
    print()

first_snapshot_ts = ms_to_ts(snapshots[0]['timestamp-ms'] + 1000) if snapshots else None
latest_snapshot_ts = ms_to_ts(snapshots[-1]['timestamp-ms'] + 1000) if snapshots else None

print(f'--- Jinja 変数（+1s offset for post-commit）---')
print(f'first_snapshot_ts  = {first_snapshot_ts}')
print(f'latest_snapshot_ts = {latest_snapshot_ts}')

## STEP 3: タイムトラベル（スナップショット指定）

Iceberg Table では `AT (SNAPSHOT => '<snapshot_id>')` または  
`AT (TIMESTAMP => '<timestamp>')` で過去時点のデータを参照できる。

In [ ]:
%%sql -r snapshot_count
-- 最初のスナップショット（1回目INSERT直後）を参照
SELECT COUNT(*) AS row_count_at_first_snapshot
FROM ICEBERG_DB.WORK.ORDERS_ICEBERG
AT (TIMESTAMP => '{{first_snapshot_ts}}'::TIMESTAMP_LTZ);

In [ ]:
%%sql -r timestamp_count
-- タイムスタンプ指定でタイムトラベル（最新スナップショット時点）
SELECT COUNT(*) AS row_count_at_latest
FROM ICEBERG_DB.WORK.ORDERS_ICEBERG
AT (TIMESTAMP => '{{latest_snapshot_ts}}'::TIMESTAMP_LTZ);

## STEP 4: 変更履歴の確認（CHANGES 句）

`CHANGES` 句を使うと、2つの時点間で変更された行を取得できる。  
利用するには事前に `CHANGE_TRACKING = TRUE` を設定する必要がある。

**CHANGE_TRACKING とは：**
- テーブルに対する INSERT / UPDATE / DELETE の変更メタデータを Snowflake が内部的に記録する仕組み
- 有効化すると、`CHANGES` 句・Stream が利用可能になる
- Iceberg テーブルでは `ALTER ICEBERG TABLE ... SET CHANGE_TRACKING = TRUE` で有効化（`ALTER TABLE` は不可）
- 有効化した**時点以降**の変更のみ追跡される（過去に遡っての追跡は不可）
- 追加のストレージコストが発生する（変更メタデータの保持分）

In [ ]:
%%sql -r change_tracking_result
-- Iceberg Table に CHANGE TRACKING を有効化
ALTER ICEBERG TABLE ICEBERG_DB.WORK.ORDERS_ICEBERG SET CHANGE_TRACKING = TRUE;

In [ ]:
%%sql -r changes_result
-- CHANGE_TRACKING 有効化後の変更を確認
-- ※ CHANGE_TRACKING は有効化した時点以降の変更のみ追跡可能
-- ※ 有効化前のスナップショット時点を指定するとエラーになる
-- ※ ここでは有効化直後のため変更データは空（0件）が正常
SELECT
    METADATA$ACTION,
    METADATA$ISUPDATE,
    O_ORDERKEY,
    O_ORDERSTATUS
FROM ICEBERG_DB.WORK.ORDERS_ICEBERG
CHANGES(INFORMATION => APPEND_ONLY)
AT (TIMESTAMP => '{{latest_snapshot_ts}}'::TIMESTAMP_LTZ)
LIMIT 20;

## STEP 5: Iceberg Table のメタデータ詳細確認

In [ ]:
%%sql -r iceberg_info
-- テーブル情報の詳細（S3 上のメタデータパスが確認できる）
SELECT TRY_PARSE_JSON(
    SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_DB.WORK.ORDERS_ICEBERG')
) AS info;

In [ ]:
%%sql -r iceberg_metadata
-- メタデータの各フィールドを展開して確認
WITH info AS (
    SELECT TRY_PARSE_JSON(
        SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_DB.WORK.ORDERS_ICEBERG')
    ) AS j
)
SELECT
    j:status::STRING            AS status,
    j:metadataLocation::STRING  AS metadata_location
FROM info;

## STEP 6: テーブル統計の確認

In [ ]:
%%sql -r table_stats
-- テーブルのストレージ使用量
SELECT
    TABLE_NAME,
    ROW_COUNT,
    BYTES,
    CLUSTERING_KEY,
    CREATED
FROM ICEBERG_DB.INFORMATION_SCHEMA.TABLES
WHERE TABLE_NAME = 'ORDERS_ICEBERG';

## まとめ・気づき

- `AT (SNAPSHOT => ...)` でスナップショット単位のタイムトラベルが可能
- `AT (TIMESTAMP => ...)` でタイムスタンプ指定のタイムトラベルも可能
- `CHANGES` 句で2時点間の差分（INSERT/DELETE/UPDATE）を取得できる
- `SYSTEM$ICEBERG_TABLE_INFO()` で S3 上の metadata ファイルの場所が確認できる
- Snowflake Managed Iceberg は Snowflake が catalog（スナップショット管理）を担当する